In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path 


In [2]:
# Labels for answer choices the data leaves without one: a bare number ('02') or no value label at all (NaN).
# A scale is identified by its labelled choices, in code order; its entry gives the label of every unlabelled code.
# The same scale gets the same labels in any dataset. Labelled choices are never changed.
LABEL_BANK = {
    ("None", "1 child", "2 children"): {
        3: "3 children",
        4: "4 children",
        5: "5 children",
        6: "6 children",
        7: "7 children",
        8: "8 children",
        9: "9 children",
        10: "10 children",
    },
    ("Yes, they definitely should", "No, they definitely should not"): {
        2: "Yes, they should",
        3: "Yes, they probably should",
        4: "Undecided",
        5: "No, they probably should not",
        6: "No, they should not",
    },
    ("You cannot rely on either of them", "You can definitely rely more on close relatives",
     "You can rely equally on both", "You can definitely rely more on close friends"): {
        2: "You can rely more on close relatives",
        3: "You can probably rely more on close relatives",
        5: "You can probably rely more on close friends",
        6: "You can rely more on close friends",
    },
    ("Close relatives should definitely be the most important", "Both should be equally important",
     "Close friends should definitely be the most important"): {
        2: "Close relatives should be the most important",
        3: "Close relatives should probably be the most important",
        5: "Close friends should probably be the most important",
        6: "Close friends should be the most important",
    },
}

def unlabelled(text):
    return not isinstance(text, str) or not re.search(r"[A-Za-z]", text)   # NaN or no letters ('02')

def scale_key(texts):
    return tuple(t for t in texts if not unlabelled(t))                      # labelled choices in code order

def fill_labels(codes, texts):
    fixes = LABEL_BANK.get(scale_key(texts), {})
    return [fixes.get(c, t) if unlabelled(t) else t for c, t in zip(codes, texts)]


COUNTRY_NOTE = re.compile(r"^([A-Z]{2}(?:\s*,\s*[A-Z]{2})*)(?:\s+also)?\s*:\s*")   # 'NL, US: Employers' -> offered only in NL and US

def offered_options(labels, country, verbose=False):
    """{code: text} of every valid (non-negative) option offered in `country`, in code order. An option marked for
    other countries only is dropped; an option marked for `country` keeps its text without the marker.
    verbose=True prints every option it drops or changes."""
    out = {}
    for k, t in sorted(labels.items()):
        if k < 0:
            continue
        text = re.sub(r"^-?\d+\.\s*", "", t)
        m = COUNTRY_NOTE.match(text)
        if m:
            if country not in re.split(r"\s*,\s*", m[1]):
                if verbose:
                    print(f"  {int(k):>4}  {text!r}  ->  dropped: not offered in {country}")
                continue
            if verbose:
                print(f"  {int(k):>4}  {text!r}  ->  {text[m.end():]!r}")
            text = text[m.end():]
        out[int(k)] = text
    return out

def persona_text(label, country):
    """Persona value as this country's respondents saw it: '1. Male' -> 'Male'; 'Once a year; US: Once or twice a year'
    -> 'Once or twice a year' in the US, 'Once a year' elsewhere."""
    text = re.sub(r"^-?\d+\.\s+", "", str(label)).strip()
    m = re.fullmatch(r"(.*?)\s*\(([A-Z]{2}(?:\s*,\s*[A-Z]{2})*)\s*:\s*(.*)\)", text)   # a note in brackets: '89 (US: 89 years or older)'
    if m:
        return m[3] if country in re.split(r"\s*,\s*", m[2]) else m[1]
    kept, ours = [], None
    for part in re.split(r";|(?<![A-Z]{2}),\s*(?=[A-Z]{2}(?:\s*,\s*[A-Z]{2})*\s*:)", text):
        part = part.strip()
        m = COUNTRY_NOTE.match(part)
        if not m:
            kept.append(part)                          # general wording
        elif country in re.split(r"\s*,\s*", m[1]):
            ours = part[m.end():]                      # this country's own wording
            if re.match(r"(incl|\+|also)", ours, re.I):  # a note that adds to the general wording: 'incl. nearly every week'
                ours = f"{kept[0]}, {ours.lstrip('+ ')}" if kept else ours
    return ours if ours is not None else "; ".join(kept)

In [3]:
data_dir = Path.cwd()/'data/issp2022'
data_fn = data_dir/'raw/ZA10000_v2-0-0.dta'
bank_fn = data_dir/'question_bank.csv'
verified_fn = data_dir/'questions_verified.csv'

country = 'US'
max_options = 10

bank = pd.read_csv(bank_fn)

wide = pd.read_stata(data_fn)                                   # as stored; answers shown with GESIS's labels
raw = pd.read_stata(data_fn, convert_categoricals=False)        # same file, answers as integer codes
labels = pd.io.stata.StataReader(data_fn).variable_labels()     # column -> GESIS label
with pd.io.stata.StataReader(data_fn) as reader:
    label_sets = reader.value_labels()                                   # label-set name -> {code: "N. text"}
    value_labels = {v: label_sets[s] if s in label_sets else {} for v, s in zip(reader._varlist, reader._lbllist)}   # column -> {code: "N. text"}

wide = wide[wide["c_alphan"] == country]
raw = raw.loc[wide.index]                                       # same rows as wide
wide["CASEID"] = wide["CASEID"].astype("int64")                 # stored as a float; shown as 2.022008e+15 otherwise

persona = ["c_alphan", "SEX", "AGE", "EDULEVEL", "RELIGGRP", "ATTEND", "PARTY_LR", "VOTE_LE", "URBRURAL",
           "MARITAL", "MAINSTAT", "UNION", f"{country}_PRTY", f"{country}_INC", f"{country}_REG", f"{country}_ETHN1"]
questions = [c for c in wide.columns if re.fullmatch(r"v\d+", c)]   # v1, v2, ...: the substantive questions

print(f"{country} respondents: {len(wide)}, persona cols: {len(persona)}, raw question cols: {len(questions)}")

US respondents: 1126, persona cols: 16, raw question cols: 72


In [4]:
options = {v: offered_options(value_labels[v], country) for v in questions}   # v -> {code: text}, every option offered in this country

wide[questions] = wide[questions].where(raw[questions] >= 0)                          # negative answers -> NaN (that cell only)

questions = [v for v in questions                                              # the one place questions are kept or dropped:
             if v in set(bank.loc[bank["type"] != "exclude", "v"])            #   in the bank and not marked exclude
             and len(options[v]) <= max_options]                              #   at most max_options offered options

bank = bank[bank["v"].isin(questions)]
wide = wide[["CASEID"] + persona + questions]

wide[questions] = wide[questions].apply(lambda s: s.cat.set_categories(
    [value_labels[s.name][k] for k in options[s.name]]))                # categories = every option offered in this country, in code order
print(f"Kept {len(questions)} substantive questions (<= {max_options} options, not excluded)")

Kept 64 substantive questions (<= 10 options, not excluded)


In [5]:
# persona: number prefix removed, this country's wording taken from country notes, negative codes -> NaN
for c in persona[1:]:                                             # c_alphan stays as it is
    vl = value_labels[c]
    values = {}
    for k in sorted(raw[c].dropna().unique()):
        raw_label = int(k) if c == "AGE" and k >= 0 else vl[k]
        before = re.sub(r"^-?\d+\.\s+", "", str(raw_label)).strip()
        if k < 0:
            values[k] = float("nan")
            print(f"{c:<10} {int(k):>6}  {before!r}  ->  NaN  ({int((raw[c] == k).sum())} respondents)")
        else:
            values[k] = persona_text(raw_label, country)
            if values[k] != before:
                print(f"{c:<10} {int(k):>6}  {before!r}  ->  {values[k]!r}")
    wide[c] = [values[k] if pd.notna(k) else float("nan") for k in raw[c]]

SEX            -9  'No answer'  ->  NaN  (4 respondents)
AGE            -9  'No answer'  ->  NaN  (57 respondents)
RELIGGRP       -9  'No answer'  ->  NaN  (14 respondents)
ATTEND         -9  'No answer'  ->  NaN  (6 respondents)
ATTEND          6  'Once a year; US: Once or twice a year'  ->  'Once or twice a year'
PARTY_LR       -9  'No answer'  ->  NaN  (120 respondents)
PARTY_LR       -4  'NAP (code 2, -4, -7 in VOTE_LE)'  ->  NaN  (337 respondents)
VOTE_LE        -9  'No answer'  ->  NaN  (80 respondents)
VOTE_LE        -4  'Not eligible to vote in last election'  ->  NaN  (125 respondents)
MARITAL        -9  'No answer'  ->  NaN  (5 respondents)
MAINSTAT       -9  'No answer'  ->  NaN  (1 respondents)
UNION          -1  'Not available (US)'  ->  NaN  (1126 respondents)
US_PRTY        -9  'No answer'  ->  NaN  (120 respondents)
US_PRTY        -4  'NAP (code 2, -4, -7 in VOTE_LE)'  ->  NaN  (337 respondents)
US_INC         -9  'No answer'  ->  NaN  (129 respondents)
US_ETHN1       -

In [6]:
long = wide.melt(id_vars=["CASEID"]+persona, value_vars=questions, var_name="v", value_name="answer")
long.insert(long.columns.get_loc("v") + 1, "label", [labels[v] for v in long["v"]])
long["answer_code"] = raw[questions].melt()["value"].astype("Int64").to_numpy()   # code from the raw integers, not the label string
long = long.dropna(subset=["answer"])  # drop the NaN cells (negative or missing answers)

long.head(2)

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,MAINSTAT,UNION,US_PRTY,US_INC,US_REG,US_ETHN1,v,label,answer,answer_code
0,2022008400000002,US,Male,80,"Upper level tertiary, MA",Catholic,Once or twice a year,Center / liberal,Yes,The suburbs or outskirts of a big city,...,Retired,NaN,Democrat,NaN,New England,Austria,v1,Q1a Working mom can establish as warm relation...,3. Neither agree nor disagree,3
1,2022008400000004,US,Female,23,"Lower level tertiary, BA",No religion,Once a month,NaN,NaN,The suburbs or outskirts of a big city,...,In paid work,NaN,NaN,90 000 - 109 999 USD,New England,England and Wales,v1,Q1a Working mom can establish as warm relation...,2. Agree,2


In [7]:
# bank: every option offered in this country, in code order; label fixes shown before -> after
info = bank.set_index("v")
rows = []
for v in questions:
    before = {int(k): re.sub(r"^-?\d+\.\s*", "", t) for k, t in sorted(value_labels[v].items()) if k >= 0}   # as the data has it
    opts = options[v]                                                                                       # country markers handled
    codes = list(opts)
    after = dict(zip(codes, fill_labels(codes, list(opts.values()))))                                       # unlabelled choices filled
    rows.append({"v": v,
                 "answer_code": codes,
                 "answer": [value_labels[v][k] for k in codes],
                 "answer_text": list(after.values())})
    if before == after:
        continue
    print("=" * 100)
    print(f"{v} | {info.loc[v, 'type']} | {info.loc[v, 'data_label']}")
    if any(unlabelled(t) for t in after.values()):
        print("NO ENTRY IN LABEL_BANK FOR THIS SCALE")
    print("-" * 100)
    for c, b in before.items():
        print(f"{c:>4}  {b:<45} ->  {after.get(c, f'(dropped: not offered in {country})')}")
print("=" * 100)

bank = bank.merge(pd.DataFrame(rows), on="v")

# long: each person's answer text taken from bank, so the two always agree
text_of = {(r["v"], c): t for _, r in bank.iterrows() for c, t in zip(r["answer_code"], r["answer_text"])}
long["answer_text"] = [text_of[(v, c)] for v, c in zip(long["v"], long["answer_code"])]

# long: the question each answer belongs to, with its type, direction and every option offered
long = long.merge(
    bank[["v", "question", "type", "reversed", "answer_code", "answer_text"]]
        .rename(columns={"answer_code": "option_codes", "answer_text": "option_texts"}),
    on="v", how="left")
long.head(2)

v9 | choice | Q3a Should women work: child under school age
----------------------------------------------------------------------------------------------------
   1  Work full-time                                ->  Work full-time
   2  Work part-time                                ->  Work part-time
   3  Stay at home                                  ->  Stay at home
   4  TW: Women should decide                       ->  (dropped: not offered in US)
v10 | choice | Q3b Should women work: youngest kid at school
----------------------------------------------------------------------------------------------------
   1  Work full-time                                ->  Work full-time
   2  Work part-time                                ->  Work part-time
   3  Stay at home                                  ->  Stay at home
   4  TW: Women should decide                       ->  (dropped: not offered in US)
v32 | choice | Q13 Elderly people: provider of domestic help
------------------------

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,v,label,answer,answer_code,answer_text,question,type,reversed,option_codes,option_texts
0,2022008400000002,US,Male,80,"Upper level tertiary, MA",Catholic,Once or twice a year,Center / liberal,Yes,The suburbs or outskirts of a big city,...,v1,Q1a Working mom can establish as warm relation...,3. Neither agree nor disagree,3,Neither agree nor disagree,"To begin, we have some questions about women. ...",score,True,"[1, 2, 3, 4, 5]","[Strongly agree, Agree, Neither agree nor disa..."
1,2022008400000004,US,Female,23,"Lower level tertiary, BA",No religion,Once a month,NaN,NaN,The suburbs or outskirts of a big city,...,v1,Q1a Working mom can establish as warm relation...,2. Agree,2,Agree,"To begin, we have some questions about women. ...",score,True,"[1, 2, 3, 4, 5]","[Strongly agree, Agree, Neither agree nor disa..."


In [8]:
missing = bank[bank["answer_text"].apply(
    lambda texts: any(not isinstance(t, str) or not re.search(r"[A-Za-z]", t) for t in texts))]
missing[["v", "type", "data_label", "answer_code", "answer_text"]]


,v,type,data_label,answer_code,answer_text


### Check persona

In [9]:
EDULEVEL_MAP = {
    "No (formal) education, incomplete primary": "Less than high school",
    "Primary education": "Less than high school",
    "Lower secondary": "Less than high school",
    "Upper secondary": "Upper secondary",
    "Short-cycle tertiary": "Short-cycle tertiary",
    "Lower level tertiary, BA": "College graduate/some postgrad",
    "Upper level tertiary, MA": "College graduate/some postgrad",
    "PhD, Post tertiary specialization": "College graduate/some postgrad",
}
if set(long["EDULEVEL"].dropna().unique()) != set(EDULEVEL_MAP.keys()):
    raise ValueError(f"Unexpected EDULEVEL values: {set(long['EDULEVEL'].dropna().unique()) ^ set(EDULEVEL_MAP.keys())}")
long["EDULEVEL"] = [EDULEVEL_MAP[v] if pd.notna(v) else v for v in long["EDULEVEL"]]
long["EDULEVEL"].value_counts(dropna=False)

EDULEVEL
Upper secondary                   21691
College graduate/some postgrad    19011
Less than high school              5934
Short-cycle tertiary               4667
Name: count, dtype: int64

In [10]:
RELIGGRP_MAP = {
    "No religion": "Atheist",
    "Islamic": "Muslim",
    "Protestant": "Protestant",
    "Catholic": "Catholic",
    "Orthodox": "Orthodox",
    "Other Christian": "Other Christian",
    "Jewish": "Jewish",
    "Buddhist": "Buddhist",
    "Hindu": "Hindu",
    "Other Asian Religions": "Other Asian Religions",
    "Other Religions": "Other Religions",
}
if set(long["RELIGGRP"].dropna().unique()) != set(RELIGGRP_MAP.keys()):
    raise ValueError(f"Unexpected RELIGGRP values: {set(long['RELIGGRP'].dropna().unique()) ^ set(RELIGGRP_MAP.keys())}")
long["RELIGGRP"] = [RELIGGRP_MAP[v] if pd.notna(v) else v for v in long["RELIGGRP"]]
long["RELIGGRP"].value_counts(dropna=False)

RELIGGRP
Protestant               19665
Atheist                  14797
Catholic                 11740
Other Christian           1272
Other Religions           1012
Jewish                     996
NaN                        614
Buddhist                   349
Muslim                     325
Orthodox                   285
Hindu                      146
Other Asian Religions      102
Name: count, dtype: int64

In [11]:
PARTY_LR_MAP = {
    "Center / liberal": "Liberal",
    "Right / conservative": "Conservative",
    "Other": "Other",
}
if set(long["PARTY_LR"].dropna().unique()) != set(PARTY_LR_MAP.keys()):
    raise ValueError(f"Unexpected PARTY_LR values: {set(long['PARTY_LR'].dropna().unique()) ^ set(PARTY_LR_MAP.keys())}")
long["PARTY_LR"] = [PARTY_LR_MAP[v] if pd.notna(v) else v for v in long["PARTY_LR"]]
long["PARTY_LR"].value_counts(dropna=False)

PARTY_LR
NaN             20620
Liberal         16410
Conservative    11986
Other            2287
Name: count, dtype: int64

In [12]:
US_PRTY_MAP = {
    "Democrat": "Democrat",
    "Republican": "Republican",
    "Other candidate": "Other candidate",
}
if set(long["US_PRTY"].dropna().unique()) != set(US_PRTY_MAP.keys()):
    raise ValueError(f"Unexpected US_PRTY values: {set(long['US_PRTY'].dropna().unique()) ^ set(US_PRTY_MAP.keys())}")
long["US_PRTY"] = [US_PRTY_MAP[v] if pd.notna(v) else v for v in long["US_PRTY"]]
long["US_PRTY"].value_counts(dropna=False)

US_PRTY
NaN                20620
Democrat           16410
Republican         11986
Other candidate     2287
Name: count, dtype: int64

In [13]:
US_INC_MAP = {
    "Under 1 000 USD per year, before taxes": "Less than $30,000",
    "1 000 - 2 999 USD": "Less than $30,000",
    "3 000 - 3 999 USD": "Less than $30,000",
    "5 000 - 5 999 USD": "Less than $30,000",
    "6 000 - 6 999 USD": "Less than $30,000",
    "7 000 - 7 999 USD": "Less than $30,000",
    "8 000 - 9 999 USD": "Less than $30,000",
    "10 000 - 12 499 USD": "Less than $30,000",
    "12 500 - 14 999 USD": "Less than $30,000",
    "15 000 - 17 499 USD": "Less than $30,000",
    "17 500 - 19 999 USD": "Less than $30,000",
    "20 000 - 22 499 USD": "Less than $30,000",
    "22 500 - 24 999 USD": "Less than $30,000",
    "25 000 - 29 999 USD": "Less than $30,000",
    "30 000 - 34 999 USD": "$30,000 - $34,999",
    "35 000 - 39 999 USD": "$35,000 - $39,999",
    "40 000 - 49 999 USD": "$40,000 - $49,999",
    "50 000 - 59 999 USD": "$50,000 - $59,999",
    "60 000 - 74 999 USD": "$60,000 - $74,999",
    "75 000 - 89 999 USD": "$75,000 - $89,999",
    "90 000 - 109 999 USD": "$90,000 - $109,999",
    "110 000 - 129 999 USD": "$100,000 or more",
    "130 000 - 149 999 USD": "$100,000 or more",
    "150 000 - 169 999 USD": "$100,000 or more",
    "170 000 USD or more per year, before taxes": "$100,000 or more",
}
if set(long["US_INC"].dropna().unique()) != set(US_INC_MAP.keys()):
    raise ValueError(f"Unexpected US_INC values: {set(long['US_INC'].dropna().unique()) ^ set(US_INC_MAP.keys())}")
long["US_INC"] = [US_INC_MAP[v] if pd.notna(v) else v for v in long["US_INC"]]
long["US_INC"].value_counts(dropna=False)

US_INC
$100,000 or more      11808
Less than $30,000     10077
NaN                    5488
$60,000 - $74,999      4964
$75,000 - $89,999      3960
$90,000 - $109,999     3907
$50,000 - $59,999      3479
$40,000 - $49,999      3374
$30,000 - $34,999      2454
$35,000 - $39,999      1792
Name: count, dtype: int64

In [14]:
US_REG_MAP = {
    "New England": "Northeast",
    "Middle Atlantic": "Northeast",
    "South Atlantic": "South",
    "East South Central": "South",
    "West South Central": "South",
    "East North Central": "East North Central",
    "West North Central": "West North Central",
    "Mountain": "Mountain",
    "Pacific": "Pacific",
}
if set(long["US_REG"].dropna().unique()) != set(US_REG_MAP.keys()):
    raise ValueError(f"Unexpected US_REG values: {set(long['US_REG'].dropna().unique()) ^ set(US_REG_MAP.keys())}")
long["US_REG"] = [US_REG_MAP[v] if pd.notna(v) else v for v in long["US_REG"]]
long["US_REG"].value_counts(dropna=False)

US_REG
South                 21324
Pacific                7651
Northeast              7523
East North Central     7360
Mountain               4091
West North Central     3354
Name: count, dtype: int64

In [15]:
ATTEND_MAP = {
    "Several times a week or more often": "Several times a week or more often",
    "Once a week": "Once a week",
    "2 or 3 times a month": "2 or 3 times a month",
    "Once a month": "Once a month",
    "Several times a year": "Several times a year",
    "Once or twice a year": "Once or twice a year",
    "Less frequently than once a year": "Less frequently than once a year",
    "Never": "Never",
}
if set(long["ATTEND"].dropna().unique()) != set(ATTEND_MAP.keys()):
    raise ValueError(f"Unexpected ATTEND values: {set(long['ATTEND'].dropna().unique()) ^ set(ATTEND_MAP.keys())}")
long["ATTEND"] = [ATTEND_MAP[v] if pd.notna(v) else v for v in long["ATTEND"]]
long["ATTEND"].value_counts(dropna=False)

ATTEND
Never                                 18050
Once or twice a year                   6818
Once a week                            6411
2 or 3 times a month                   5379
Less frequently than once a year       5038
Several times a year                   4987
Once a month                           2226
Several times a week or more often     2144
NaN                                     250
Name: count, dtype: int64

In [16]:
MARITAL_MAP = {
    "Married": "Married",
    "Separated from spouse/civil partner (still legally married/still legally in a civil partnership)": "Separated from spouse/civil partner (still legally married/still legally in a civil partnership)",
    "Divorced from spouse/legally separated from civil partner": "Divorced from spouse/legally separated from civil partner",
    "Widowed/civil partner died": "Widowed/civil partner died",
    "Never married/never in a civil partnership": "Never married/never in a civil partnership",
}
if set(long["MARITAL"].dropna().unique()) != set(MARITAL_MAP.keys()):
    raise ValueError(f"Unexpected MARITAL values: {set(long['MARITAL'].dropna().unique()) ^ set(MARITAL_MAP.keys())}")
long["MARITAL"] = [MARITAL_MAP[v] if pd.notna(v) else v for v in long["MARITAL"]]
long["MARITAL"].value_counts(dropna=False)

MARITAL
Married                                                                                             23165
Never married/never in a civil partnership                                                          16056
Divorced from spouse/legally separated from civil partner                                            7918
Widowed/civil partner died                                                                           2687
Separated from spouse/civil partner (still legally married/still legally in a civil partnership)     1264
NaN                                                                                                   213
Name: count, dtype: int64

In [17]:
UNION_MAP = {}
if set(long["UNION"].dropna().unique()) != set(UNION_MAP.keys()):
    raise ValueError(f"Unexpected UNION values: {set(long['UNION'].dropna().unique()) ^ set(UNION_MAP.keys())}")
long["UNION"] = [UNION_MAP[v] if pd.notna(v) else v for v in long["UNION"]]
long["UNION"].value_counts(dropna=False)

UNION
NaN    51303
Name: count, dtype: int64

In [18]:
US_ETHN1_MAP = {
    "Netherlands (Dutch/Holland)": "Netherlands (Dutch / Holland)",
}
missing_ethn = set(US_ETHN1_MAP.keys()) - set(long["US_ETHN1"].dropna().unique())
if missing_ethn:
    raise ValueError(f"Expected US_ETHN1 keys not found: {missing_ethn}")
long["US_ETHN1"] = [US_ETHN1_MAP[v] if v in US_ETHN1_MAP else v for v in long["US_ETHN1"]]
long["US_ETHN1"].value_counts(dropna=False).head(15)

US_ETHN1
England and Wales       11290
Germany                  6445
NaN                      5105
Mexico                   3210
Ireland                  3144
France                   1931
Italy                    1518
Other African            1201
American Indian          1126
Other North American     1113
Czechoslovakia           1049
Denmark                  1020
Spain                     983
Poland                    924
Austria                   890
Name: count, dtype: int64

In [19]:
bad_persona = {}
for c in persona:
    invalid = [
        v for v in long[c].dropna().unique()
        if re.search(r";|\b[A-Z]{2}(?:\s*,\s*[A-Z]{2})*\s*:|^-?\d+\.\s+|\b(No answer|Don't know|Refused|NAP|NAV|Not available|Can't choose)\b", str(v), re.I)
    ]
    if invalid:
        bad_persona[c] = invalid
if set(long["c_alphan"].unique()) != {"US"}:
    bad_persona["c_alphan"] = list(long["c_alphan"].unique())
if set(long["SEX"].dropna().unique()) != {"Male", "Female"}:
    bad_persona["SEX"] = list(long["SEX"].dropna().unique())
if set(long["VOTE_LE"].dropna().unique()) != {"Yes", "No"}:
    bad_persona["VOTE_LE"] = list(long["VOTE_LE"].dropna().unique())
if bad_persona:
    raise ValueError(f"Unnormalized persona values found: {bad_persona}")

# Export verified

In [20]:
# ---- export for training: one row per answer, with the person's persona, the question and every option offered
long_fn = data_dir/f"issp2022_{country}_long.parquet"
columns = ["CASEID"] + persona + ["v", "label", "question", "type", "reversed", "option_codes", "option_texts",
                                  "answer_code", "answer_text"]
long[columns].to_parquet(long_fn, index=False)
long_fn

PosixPath('/home/dw/github/decision-models-finetuning/code/data/issp2022/issp2022_US_long.parquet')

In [21]:
bank['options_old'] = bank['options']
bank[['v', 'data_label', 'question', 'options_old', 'answer_code', 'answer_text', 'type', 'reversed']].to_csv(verified_fn,index=False)
